# Premier League Player Performance Evaluation Framework

This notebook implements a position-aware percentile evaluation model for Premier League players, incorporating principles from modern sports analytics (such as Gradient Sports).

### Key Methodology Pillars:
1. **Transfer Aggregation**: Merges multi-club stints for mid-season transfers into 1 player record per season.
2. **True Team Defensive Strength**: Computes Team Goals Conceded (`team_ga90`) and Clean Sheet % (`team_cs_pct`) directly from match & goalkeeper data.
3. **Position-Aware Percentile Normalization**: Evaluates players within 5 position groups (`DF`, `CM_DM`, `AM_WG`, `FW`, `GK`).
4. **Volume + Rate + Sample Reliability**: Combines total output volume (goals/assists/tackles), per-90 rates, and minimum playing time thresholds (`90s >= 10.0`).

In [ ]:
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Step 1: Load Datasets with UTF-8 Encoding

In [ ]:
processed_dir = os.path.join("..", "data", "processed")
if not os.path.exists(processed_dir):
    processed_dir = os.path.join("data", "processed")

players_df = pd.read_csv(os.path.join(processed_dir, "players.csv"), encoding='utf-8-sig')
shooting_df = pd.read_csv(os.path.join(processed_dir, "shooting.csv"), encoding='utf-8-sig')
misc_df = pd.read_csv(os.path.join(processed_dir, "miscellaneous_stats.csv"), encoding='utf-8-sig')
gk_df = pd.read_csv(os.path.join(processed_dir, "goalkeeping.csv"), encoding='utf-8-sig')

def aggregate_player_records(df, num_cols):
    """Aggregates stats for players with multiple team entries in a single season."""
    if 'Player' not in df.columns:
        return df
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            
    group_col = 'player_id' if 'player_id' in df.columns else 'Player'
    agg_dict = {col: 'sum' for col in num_cols if col in df.columns}
    for c in ['Player', 'Pos', 'Nation', 'Squad']:
        if c in df.columns:
            agg_dict[c] = 'last'
            
    return df.groupby(group_col).agg(agg_dict).reset_index(drop=True)

clean_players = aggregate_player_records(players_df, ['Min', '90s', 'Gls', 'Ast', 'G+A', 'CrdY', 'CrdR'])
clean_shooting = aggregate_player_records(shooting_df, ['Sh', 'SoT', 'Gls', 'PK', 'PKatt'])
clean_misc = aggregate_player_records(misc_df, ['CrdY', 'CrdR', 'Fls', 'Fld', 'Int', 'TklW', 'OG'])

print(f"Consolidated outfield players: {len(clean_players)}")

## Step 2: Calculate True Team Defensive Performance

In [ ]:
gk_df['GA'] = pd.to_numeric(gk_df['GA'], errors='coerce').fillna(0)
gk_df['CS'] = pd.to_numeric(gk_df['CS'], errors='coerce').fillna(0)
gk_df['90s'] = pd.to_numeric(gk_df['90s'], errors='coerce').fillna(0)

team_gk = gk_df.groupby('Squad').agg({'GA': 'sum', '90s': 'max', 'CS': 'sum'}).reset_index()
team_gk['team_ga90'] = (team_gk['GA'] / team_gk['90s'].replace(0, np.nan)).fillna(1.5)
team_gk['team_cs_pct'] = ((team_gk['CS'] / team_gk['90s'].replace(0, np.nan)).fillna(0)) * 100

print("Top 5 Defensive Teams (Lowest GA per 90):")
print(team_gk.sort_values('team_ga90')[['Squad', 'GA', 'team_ga90', 'team_cs_pct']].head(5).to_string(index=False))

## Step 3: Merge Metrics & Compute Per-90 Rates

In [ ]:
df = pd.merge(clean_players, clean_shooting[['Player', 'Sh', 'SoT']], on='Player', how='left')
df = pd.merge(df, clean_misc[['Player', 'Fls', 'Fld', 'Int', 'TklW']], on='Player', how='left').fillna(0)
df = pd.merge(df, team_gk[['Squad', 'team_ga90', 'team_cs_pct']], on='Squad', how='left')

df['90s'] = pd.to_numeric(df['90s'], errors='coerce').replace(0, np.nan)
df['ga_total'] = df['Gls'] + df['Ast']
df['ga_per90'] = (df['ga_total'] / df['90s']).fillna(0)
df['tkl_int_total'] = df['TklW'] + df['Int']
df['tkl_int_per90'] = (df['tkl_int_total'] / df['90s']).fillna(0)
df['discipline_points'] = df['CrdY'] + (df['CrdR'] * 3)

# Sample size threshold: min 10.0 90s (~900 minutes / 10 matches)
qualified = df[df['90s'] >= 10.0].copy()

## Step 4: Robust 5-Position Classification

In [ ]:
def classify_position(pos_str):
    pos = str(pos_str).upper()
    if 'GK' in pos:
        return 'GK'
    elif 'DF' in pos:
        return 'DF'
    elif 'MF' in pos:
        if 'FW' in pos or 'AM' in pos or 'WING' in pos:
            return 'AM_WG'
        return 'CM_DM'
    elif 'FW' in pos:
        return 'FW'
    return 'CM_DM'

qualified['pos_group'] = qualified['Pos'].apply(classify_position)
print(qualified['pos_group'].value_counts())

## Step 5: Positional Percentile Model Execution

In [ ]:
def calc_percentile_rank(series, invert=False):
    """Calculates percentile rank (0-100) within positional peer group."""
    if len(series) <= 1:
        return pd.Series(50.0, index=series.index)
    ranks = series.rank(pct=True) * 100.0
    if invert:
        return 100.0 - ranks
    return ranks

scored_chunks = []

for pgroup, group in qualified.groupby('pos_group'):
    g = group.copy()
    
    if pgroup == 'DF':
        g['overall_grade'] = (
            (calc_percentile_rank(g['team_ga90'], invert=True) * 0.35) +
            (calc_percentile_rank(g['team_cs_pct']) * 0.25) +
            (calc_percentile_rank(g['tkl_int_total']) * 0.20) +
            (calc_percentile_rank(g['90s']) * 0.20)
        )
    elif pgroup == 'CM_DM':
        g['overall_grade'] = (
            (calc_percentile_rank(g['tkl_int_total']) * 0.30) +
            (calc_percentile_rank(g['ga_total']) * 0.25) +
            (calc_percentile_rank(g['ga_per90']) * 0.25) +
            (calc_percentile_rank(g['90s']) * 0.20)
        )
    elif pgroup == 'AM_WG':
        g['overall_grade'] = (
            (calc_percentile_rank(g['ga_total']) * 0.40) +
            (calc_percentile_rank(g['ga_per90']) * 0.35) +
            (calc_percentile_rank(g['90s']) * 0.25)
        )
    elif pgroup == 'FW':
        g['overall_grade'] = (
            (calc_percentile_rank(g['ga_total']) * 0.40) +
            (calc_percentile_rank(g['ga_per90']) * 0.35) +
            (calc_percentile_rank(g['90s']) * 0.25)
        )
    
    scored_chunks.append(g)

# Goalkeepers: min 10 matches
if not gk_df.empty:
    gk = gk_df.copy()
    gk['MP'] = pd.to_numeric(gk['MP'], errors='coerce').fillna(0)
    gk_qual = gk[gk['MP'] >= 10].copy()
    
    save_col = 'Save%' if 'Save%' in gk_qual.columns else 'save_pct'
    cs_col = 'CS%' if 'CS%' in gk_qual.columns else 'cs_pct'
    
    save_vals = pd.to_numeric(gk_qual[save_col], errors='coerce').fillna(0)
    cs_vals = pd.to_numeric(gk_qual[cs_col], errors='coerce').fillna(0)
    ga90_vals = pd.to_numeric(gk_qual['GA90'], errors='coerce').fillna(0)
    
    p_save = calc_percentile_rank(save_vals)
    p_cs = calc_percentile_rank(cs_vals)
    p_ga90 = calc_percentile_rank(ga90_vals, invert=True)
    
    gk_qual['overall_grade'] = (p_save * 0.40) + (p_cs * 0.35) + (p_ga90 * 0.25)
    gk_qual['pos_group'] = 'GK'
    
    scored_chunks.append(gk_qual)

final_eval = pd.concat(scored_chunks, ignore_index=True)
final_eval['overall_grade'] = final_eval['overall_grade'].round(1)
print("Positional Evaluation Model completed!")

## Step 6: Validate Rankings Across All Position Groups

In [ ]:
groups_to_display = [
    ('FW', 'Centre-Forwards'),
    ('AM_WG', 'Attacking Midfielders & Wingers'),
    ('CM_DM', 'Central / Defensive Midfielders'),
    ('DF', 'Defenders'),
    ('GK', 'Goalkeepers')
]

for pgroup, pname in groups_to_display:
    pos_data = final_eval[final_eval['pos_group'] == pgroup].sort_values('overall_grade', ascending=False).reset_index(drop=True)
    pos_data['pos_rank'] = pos_data.index + 1
    print(f"\n--- Top 5 {pname} ({pgroup}) ---")
    if pgroup in ['FW', 'AM_WG']:
        cols = ['pos_rank', 'Player', 'Squad', 'Gls', 'Ast', 'ga_total', '90s', 'overall_grade']
    elif pgroup == 'CM_DM':
        cols = ['pos_rank', 'Player', 'Squad', 'tkl_int_total', 'ga_total', '90s', 'overall_grade']
    elif pgroup == 'DF':
        cols = ['pos_rank', 'Player', 'Squad', 'tkl_int_total', 'team_ga90', '90s', 'overall_grade']
    else:
        cols = ['pos_rank', 'Player', 'Squad', 'MP', 'overall_grade']
    print(pos_data[cols].head(5).to_string(index=False))